# 🍎 Apple Game RL Training

| 단계 | 위치 | 내용 |
|------|------|------|
| **Phase 1** | 로컬 PC (Go) | `sl_data.bin` 생성 |
| **Phase 2** | 이 노트북 | SL 지도학습 → `model_sl.pt` / `model_sl.onnx` |
| **Phase 3** | 이 노트북 | MaskablePPO (SB3) → `model_rl.zip` / `model_rl.onnx` |
| **Phase 4** | 로컬 PC (Go) | ModelAnA vs AnA 비교 |

### 사전 준비
1. 로컬에서 데이터 생성: `go run . -gen-sl 500 -sl-out sl_data.bin`
2. `sl_data.bin` (~228 MB) → Google Drive `내 드라이브/apple_game/` 에 업로드
3. **런타임 → 런타임 유형 변경 → T4 GPU**
4. 설정 셀 확인 후 **위에서부터 순서대로** 실행

---
## ⚙️ 설정

In [ ]:
GITHUB_URL   = "https://github.com/scheveningen361/10_apple_game.git"
DRIVE_FOLDER = "/content/drive/MyDrive/apple_game"

# Phase 2 SL
SL_EPOCHS = 50

# Phase 3 RL
RL_ITERS  = 200   # PPO 업데이트 횟수
RL_ENVS   = 4     # 병렬 환경 수 (T4: 4~8 권장)
RL_STEPS  = 2048  # 환경당 롤아웃 스텝

---
## 🔧 환경 셋업

In [ ]:
# 1. GPU 확인
import torch, os, shutil
if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    print(f"✅ GPU : {prop.name}  ({prop.total_memory/1e9:.1f} GB)")
else:
    print("⚠️  GPU 없음 — 런타임 유형을 T4로 변경하세요")

In [ ]:
# 2. Google Drive 마운트
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_FOLDER, exist_ok=True)
print("Drive 마운트 완료:", DRIVE_FOLDER)

In [ ]:
# 3. repo 클론 / 최신화
REPO_DIR = "/content/apple_game"
RL_DIR   = REPO_DIR + "/rl"

if os.path.exists(REPO_DIR + "/.git"):
    !cd {REPO_DIR} && git pull
else:
    !git clone {GITHUB_URL} {REPO_DIR}

os.chdir(RL_DIR)
print("\n--- 현재 커밋 ---")
!cd {REPO_DIR} && git log --oneline -3
print("\n--- rl/ 파일 목록 ---")
!ls -lh

In [ ]:
# 4. sl_data.bin 복사  (Drive → 작업 디렉토리)
src = f"{DRIVE_FOLDER}/sl_data.bin"
dst = f"{RL_DIR}/sl_data.bin"

if os.path.exists(dst):
    print(f"✅ sl_data.bin 이미 있음  ({os.path.getsize(dst)/1e6:.0f} MB)")
elif os.path.exists(src):
    shutil.copy(src, dst)
    print(f"✅ Drive에서 복사 완료   ({os.path.getsize(dst)/1e6:.0f} MB)")
else:
    raise FileNotFoundError(
        f"sl_data.bin 없음!\n"
        f"로컬에서 생성 후 {DRIVE_FOLDER}/ 에 업로드하세요.\n"
        f"명령: go run . -gen-sl 500 -sl-out sl_data.bin")

---
## 🎓 Phase 2: Supervised Learning

In [ ]:
# SL 학습 (50 epochs, T4 기준 AMP FP16 ~5 min)
!python train_sl.py \
    --data   sl_data.bin \
    --out    model_sl.pt \
    --epochs {SL_EPOCHS}

In [ ]:
# SL 체크포인트 확인
ckpt = torch.load('model_sl.pt', map_location='cpu')
cfg  = ckpt['config']
print(f"Best val RMSE : {ckpt['best_val'] * 170:.4f}")
print(f"Best epoch    : {ckpt['epoch'] + 1}")
print(f"Config        : {cfg}")

In [ ]:
# SL → ONNX  (Go 호환)
import sys
sys.path.insert(0, RL_DIR)
from train_sl import AppleNetSL, export_onnx

ckpt  = torch.load('model_sl.pt', map_location='cpu')
cfg   = ckpt['config']
model = AppleNetSL(channels=cfg['channels'], n_blocks=cfg['blocks'])
model.load_state_dict(ckpt['model'])
export_onnx(model, 'model_sl.onnx', device='cpu')
print(f"model_sl.onnx  {os.path.getsize('model_sl.onnx')/1e6:.1f} MB")

In [ ]:
# SL 모델 Drive 백업
for f in ['model_sl.pt', 'model_sl.onnx']:
    if os.path.exists(f):
        shutil.copy(f, f'{DRIVE_FOLDER}/{f}')
        print(f'💾 {DRIVE_FOLDER}/{f}')

---
## 🤖 Phase 3: PPO RL (MaskablePPO)

In [ ]:
# SB3 설치 (최초 1회)
!pip install -q stable-baselines3 sb3-contrib gymnasium

In [ ]:
# RL 학습
!python train_rl.py \
    --sl      model_sl.pt \
    --out     model_rl.zip \
    --iters   {RL_ITERS}   \
    --n-envs  {RL_ENVS}    \
    --n-steps {RL_STEPS}

In [ ]:
# RL 체크포인트 확인
from sb3_contrib import MaskablePPO

rl_model = MaskablePPO.load('model_rl.zip')
print(f'Policy device  : {rl_model.device}')
print(f'Total timesteps: {rl_model.num_timesteps:,}')
print('RL 모델 로드 OK ✅')

In [ ]:
# RL → ONNX  (값함수 헤드, Go nnCtxV2 호환)
from train_rl import export_onnx_value

export_onnx_value('model_rl.zip', 'model_rl.onnx')
print(f"model_rl.onnx  {os.path.getsize('model_rl.onnx')/1e6:.1f} MB")

In [ ]:
# 최종 Drive 백업
for f in ['model_rl.zip', 'model_rl.onnx']:
    if os.path.exists(f):
        shutil.copy(f, f'{DRIVE_FOLDER}/{f}')
        print(f'💾 {DRIVE_FOLDER}/{f}')
!ls -lh {DRIVE_FOLDER}

---
## 🏆 Phase 4: 로컬 평가

Drive에서 `model_rl.onnx` 다운로드 후 로컬 PC에서:
```bash
go run -tags nn . -nn-ana -n 100 -model model_rl.onnx -ort-lib onnxruntime.dll
```

| 지표 | 설명 |
|------|------|
| AnA baseline | ~131.6 (greedy 탐색) |
| ModelAnA | RL 값함수로 greedy 탐색 |
| 목표 | ModelAnA mean score > AnA |
